In [ ]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math
import os
from matplotlib.ticker import MaxNLocator

# settings
plt.rcParams.update({
        'font.size': 14,           # general font size
        'axes.titlesize': 17,      # title
        'axes.labelsize': 17,      # x and y labels
        'xtick.labelsize': 17,     # x tick labels
        'ytick.labelsize': 17,     # y tick labels
        'legend.fontsize': 14,     # legend
    })

from pathlib import Path

PLOT_DIR = Path("./plots/pdf_figures")
PLOT_DIR.mkdir(parents=True, exist_ok=True)


# Figure 3 (a)

In [ ]:
def plot_detection_vs_surface(df,
                                                fixed_mu,
                                                fixed_n_volume,
                                                fixed_probability,
                                                fixed_n_targets,
                                                fixed_n_walkers,
                                                save_path=None,
                                                error_bands=False):
    shapes = df['TargetShape'].unique()
    #shapes = ['Line']
    
    plotted_surface_values = set()
    ax = plt.gca()
    ax.ticklabel_format(style='sci', axis='y')
    

    for shape in shapes:
        subset = df[
            (df['mu'] == fixed_mu) &
            (df['n_volume'] == fixed_n_volume) &
            (df['probability'] == fixed_probability) &
            (df['n_targets'] == fixed_n_targets) &
            (df['n_walkers'] == fixed_n_walkers) &
            (df['TargetShape'] == shape)]

        if subset.empty:
            continue

        grouped = subset.groupby('surface')['detection_time'].agg(mean='mean', std='std', count='count').reset_index()
        grouped = grouped.sort_values('surface')
        plotted_surface_values.update(grouped['surface'].values)

        line, = plt.plot(grouped['surface'], grouped['mean'], marker='o', label=shape)
        if error_bands:
            sem = grouped['std'] / np.sqrt(grouped['count'])
            plt.fill_between(grouped['surface'], grouped['mean'] - sem, grouped['mean'] + sem,
                              color=line.get_color(), alpha=0.2, linewidth=0)

    # Calcola e aggiungi curve teoriche
    surface_values = sorted(plotted_surface_values)
    volume_over_surface = [2*fixed_n_volume / d for d in surface_values]
    plt.plot(surface_values, volume_over_surface, linestyle='--', color='gray', label=r'$2n/\Delta_P$')
    #volume_over_surface_2 = [10*fixed_n_volume / d for d in surface_values]

    #volume_over_length = [fixed_n_volume / (2*d) for d in surface_values]
    #plt.plot(surface_values, volume_over_length, linestyle='-.', color='orange', label=r'$n/L$')
    #length_over_surface = [fixed_n_volume / d**2 for d in surface_values]
    #plt.plot(surface_values, length_over_surface, linestyle=':', color='red', label=r'$n/L^2$')

    #plt.plot(surface_values, volume_over_surface, linestyle='--', color='gray', label=r'$4n/\Delta$')
    #plt.plot(surface_values, volume_over_surface_2, linestyle='--', color='gray', label=r'$10n/\Delta$')

    # Disable the automatic scientific notation offset
    
    # Etichette e stile
    plt.xlabel(r'$\Delta_P$')
    #plt.xlabel(r'$L$')
    plt.xscale('log')
    plt.ylabel(r'$t_{detect}(X^2)$')
    plt.ylim(top=10**7 * 1.5)
    side = round(fixed_n_volume ** (1/3))
    #plt.title(f'Detection Time vs D — μ={fixed_mu}, side={side}, n={fixed_n_walkers}, p={fixed_probability}, n_targets={fixed_n_targets}')
    #plt.xticks(surface_values)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path,  bbox_inches="tight", format='pdf')
    
    plt.show()
    
    plt.close()


In [ ]:
df = pd.read_csv('./results/detection_time_cauchy_projected_surface/review/detection_time_cauchy_projected_surface.csv')
print(f"Surface values: {df['D'].unique()}")
print(f"Diameter for the linear target: {df[df['TargetShape'] == 'Line']['D'].unique()}")
print(f"Surface for the linear target: {df[df['TargetShape'] == 'Line']['surface'].unique()}")

plot_detection_vs_surface(df,fixed_mu=2,fixed_n_volume=134217728,fixed_probability=1,fixed_n_targets=1, fixed_n_walkers=1, save_path='./plots/pdf_figures/detection_time_vs_surface.pdf')
plot_detection_vs_surface(df,fixed_mu=2,fixed_n_volume=134217728,fixed_probability=1,fixed_n_targets=1, fixed_n_walkers=1, save_path='./plots/pdf_figures/detection_time_vs_surface_errorbands.pdf', error_bands=True)


# Figure 3 (b+c+d)

In [ ]:
def plot_detection_ratio_vs_mu_for_all_surface(df, fixed_n,
                                         fixed_n_volume=None,
                                         fixed_n_targets=None,
                                         fixed_probability=None,
                                         target_shapes=None,
                                         save_path=None,
                                         parameters = None,
                                         legend_cols=2,
                                         error_bands=False):
    import matplotlib.pyplot as plt
    import numpy as np

    base_filter = (df['n_walkers'] == fixed_n)
    if fixed_n_volume is not None:
        base_filter &= (df['n_volume'] == fixed_n_volume)
    if fixed_n_targets is not None:
        base_filter &= (df['n_targets'] == fixed_n_targets)
    if fixed_probability is not None:
        base_filter &= (df['probability'] == fixed_probability)
    
    if parameters is not None:
        base_filter &= df['surface'].isin(parameters)
    else:
        base_filter &= (df['surface'] > 100)  # Considera solo D >= 2    
    subset = df[base_filter].copy()
    
    if target_shapes is not None:
        subset = subset[subset['TargetShape'].isin(target_shapes)]

    group_cols = ['mu', 'surface', 'TargetShape', 'n_walkers']
    if fixed_n_volume is not None:
        group_cols.append('n_volume')
    if fixed_n_targets is not None:
        group_cols.append('n_targets')
    if fixed_probability is not None:
        group_cols.append('probability')

    stats = subset.groupby(group_cols)['detection_time'].agg(
        detection_time_mean='mean', detection_time_std='std', detection_time_count='count'
    ).reset_index()

    plt.figure(figsize=(8, 6))
    line_styles = ['-', '--']
    unique_shapes = sorted(stats['TargetShape'].unique())
    shape_style_map = {shape: line_styles[i % len(line_styles)] for i, shape in enumerate(unique_shapes)}
    unique_diameters = sorted(stats['surface'].unique())
    colors = plt.cm.tab10.colors

    for color_idx, D_val in enumerate(unique_diameters):
        dia_data = stats[stats['surface'] == D_val]
        base_rows = dia_data[dia_data['mu'] == 2].set_index('TargetShape')
        base_mu2 = base_rows['detection_time_mean']
        if base_mu2.empty:
            continue
        for shape in unique_shapes:
            shape_data = dia_data[dia_data['TargetShape'] == shape].sort_values('mu')
            if shape_data.empty or shape not in base_mu2:
                continue
            ratio = shape_data['detection_time_mean'] / base_mu2[shape]
            line, = plt.plot(shape_data['mu'],
                     ratio,
                     color=colors[color_idx % len(colors)],
                     linestyle=shape_style_map[shape],
                     marker='o')
            if error_bands:
                sem_num = shape_data['detection_time_std'] / np.sqrt(shape_data['detection_time_count'])
                base_std = base_rows['detection_time_std'][shape]
                base_count = base_rows['detection_time_count'][shape]
                sem_den = base_std / np.sqrt(base_count)
                rel_err = np.sqrt((sem_num / shape_data['detection_time_mean'])**2 + (sem_den / base_mu2[shape])**2)
                err = ratio * rel_err
                plt.fill_between(shape_data['mu'], ratio - err, ratio + err,
                                  color=line.get_color(), alpha=0.2, linewidth=0)

    plt.axhline(1.0, color='gray', linestyle='--', linewidth=1)

    diameter_handles = [plt.Line2D([0], [0], color=colors[i % len(colors)], lw=2)
                        for i in range(len(unique_diameters))]
    diameter_labels = [rf"{d}" for d in unique_diameters]
    legend1 = plt.legend(diameter_handles, diameter_labels, framealpha=0.5, loc='upper center', ncol=legend_cols, title=r"$\Delta$")
    if len(unique_shapes) > 1:
        shape_handles = [plt.Line2D([0], [0], color='black', linestyle=shape_style_map[s], lw=2)
                        for s in unique_shapes]
        legend2 = plt.legend(shape_handles, unique_shapes, title="Shape", loc='lower right')
    plt.gca().add_artist(legend1)

    plt.xticks(sorted(stats['mu'].unique()))
    plt.xlabel(r'$\mu$')
    plt.ylabel(r'$t_{\mathrm{detect}}(X^\mu)/t_{\mathrm{detect}}(X^2)$')
    plt.grid(True)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, format='pdf')
    
    plt.show()
    plt.close()

In [ ]:
df = pd.read_csv('./results/detection_time_mu/review/detection_time_mu_ball_line.csv')

plot_detection_ratio_vs_mu_for_all_surface(df, fixed_n_targets=1, fixed_n=1, fixed_n_volume=134217728, fixed_probability=1, target_shapes=['Ball'], parameters = [8,16,32,64,128, 256, 512, 1024, 2048, 4096, 8000], legend_cols=3, save_path='./plots/pdf_figures/detection_time_ratio_vs_mu_ball.pdf')
plot_detection_ratio_vs_mu_for_all_surface(df, fixed_n_targets=1, fixed_n=1, fixed_n_volume=134217728, fixed_probability=1, target_shapes=['Ball'], parameters = [8,16,32,64,128, 256, 512, 1024, 2048, 4096, 8000], legend_cols=3, save_path='./plots/pdf_figures/detection_time_ratio_vs_mu_ball_errorbands.pdf', error_bands=True)

In [ ]:
#file_paths = glob.glob('./results/detection_time_mu/*')
#df_list = [pd.read_csv(file) for file in file_paths if os.path.isfile(file) ]
#df = pd.concat(df_list, ignore_index=True)
df = pd.read_csv('./results/detection_time_mu/review/detection_time_mu_ball_line.csv')
plot_detection_ratio_vs_mu_for_all_surface(df, fixed_n_targets=1, fixed_n=1, fixed_n_volume=134217728, fixed_probability=1, target_shapes=['Disk'], parameters = [8,16,32,64,128, 256, 512, 1024, 2048, 4096, 8000], legend_cols=3, save_path='./plots/pdf_figures/detection_time_ratio_vs_mu_disk.pdf')
plot_detection_ratio_vs_mu_for_all_surface(df, fixed_n_targets=1, fixed_n=1, fixed_n_volume=134217728, fixed_probability=1, target_shapes=['Disk'], parameters = [8,16,32,64,128, 256, 512, 1024, 2048, 4096, 8000], legend_cols=3, save_path='./plots/pdf_figures/detection_time_ratio_vs_mu_disk_errorbands.pdf', error_bands=True)

In [ ]:
df = pd.read_csv('./results/detection_time_mu/review/detection_time_mu_ball_line.csv')

plot_detection_ratio_vs_mu_for_all_surface(df, fixed_n_targets=1, fixed_n=1, fixed_n_volume=134217728, fixed_probability=1, target_shapes=['Line'], parameters = [8,16,32,64,128, 256, 512, 1024], legend_cols=2, save_path='./plots/pdf_figures/detection_time_ratio_vs_mu_line.pdf')
plot_detection_ratio_vs_mu_for_all_surface(df, fixed_n_targets=1, fixed_n=1, fixed_n_volume=134217728, fixed_probability=1, target_shapes=['Line'], parameters = [8,16,32,64,128, 256, 512, 1024], legend_cols=2, save_path='./plots/pdf_figures/detection_time_ratio_vs_mu_line_errorbands.pdf', error_bands=True)

# Figure 4 (a + b)

In [ ]:
from cycler import cycler
colors = plt.get_cmap("tab20").colors
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)

def plot_detection_time_ratio(
    df,
    n_walkers,
    n_targets,
    fixed_target_dist,
    probability,
    mu_list,
    reference_shape = "Disk",
    output_path=None,
    reference = "DiskDiameter",
    error_bands=False,
    statistic="mean"
):
    # -----------------------------------------------
    # Filter fixed parameters
    # -----------------------------------------------
    df = df[
        (df['n_walkers'] == n_walkers) &
        (df['n_targets'] == n_targets) &
        (df['fixed_target_dist'] == fixed_target_dist) &
        (df['probability'] == probability)
        ]


    if df.empty:
        print("No matching data found for the given fixed parameters.")
        return

    if reference not in df.columns:
        raise ValueError(f"Dataset must contain '{reference}'.")

    

    # -----------------------------------------------
    # First pass: compute global y-limits and store data
    # -----------------------------------------------
    y_values = []
    precomputed = {}

    for mu_val in mu_list:
        subset = df[df['mu'] == mu_val]
        if subset.empty:
            continue

        grouped = subset.groupby(
            ['TargetShape', reference, 'D']
        )['detection_time'].agg(
            detection_time=statistic, detection_time_std='std', detection_time_count='count'
        ).reset_index()

        ball_df = grouped[grouped['TargetShape'] == 'Ball_no_boundary'] \
                    .sort_values(reference).copy()
        disk_df = grouped[grouped['TargetShape'] == reference_shape] \
                    .sort_values(reference).copy()

        # Align diameters
        ball_df[reference] += 2
        disk_df[reference] += 2

        common_diam = np.intersect1d(
            ball_df[reference], disk_df[reference]
        )
        if len(common_diam) == 0:
            continue

        ball_df = ball_df[ball_df[reference].isin(common_diam)]
        disk_df = disk_df[disk_df[reference].isin(common_diam)]

        time_ratio = (
            ball_df['detection_time'].values /
            disk_df['detection_time'].values
        )

        sem_ball = ball_df['detection_time_std'].values / np.sqrt(ball_df['detection_time_count'].values)
        sem_disk = disk_df['detection_time_std'].values / np.sqrt(disk_df['detection_time_count'].values)
        rel_err = np.sqrt(
            (sem_ball / ball_df['detection_time'].values)**2 +
            (sem_disk / disk_df['detection_time'].values)**2
        )
        time_ratio_err = time_ratio * rel_err

        if mu_val == 1:
            print(f"mu = {mu_val}, time_ratio = {time_ratio}")

        y_values.extend(time_ratio)
        if error_bands:
            y_values.extend(time_ratio + time_ratio_err)
            y_values.extend(time_ratio - time_ratio_err)

        precomputed[mu_val] = (common_diam, ball_df, disk_df, time_ratio, time_ratio_err)

    if len(y_values) == 0:
        print("No valid Ball/Disk matched data for requested μ values.")
        return

    # y-limits with padding
    y_min, y_max = min(y_values), max(y_values)
    padding = 0.05 * (y_max - y_min)
    y_min -= padding
    y_max += padding

    # -----------------------------------------------
    # Single plot
    # -----------------------------------------------
    plt.figure(figsize=(8, 6))

    mu_handles = []
    theo_handle = None

    for mu_val in mu_list:
        if mu_val not in precomputed:
            continue

        common_diam, ball_df, disk_df, time_ratio, time_ratio_err = precomputed[mu_val]

        h_mu, = plt.plot(common_diam, time_ratio, 'o-', label=f"μ = {mu_val}")
        mu_handles.append(h_mu)
        if error_bands:
            plt.fill_between(common_diam, time_ratio - time_ratio_err, time_ratio + time_ratio_err,
                              color=h_mu.get_color(), alpha=0.2, linewidth=0)



        # Add theoretical ratio only for mu = 2
        if mu_val == 2: 
            if reference == "DiskDiameter":
                if reference_shape == "Disk":
                    theo_ratio = 1
                    label = r"$D^{2}_{disk}/D^{2}_{ball}$"
                elif reference_shape == "Line":
                    #theo_ratio = ( 4*np.pi + (disk_df['D'].values * np.pi * 2)) / (np.pi * ((ball_df['D'].values)**2))
                    # theo_ratio = ( (4/3)*np.pi + (disk_df['D'].values * np.pi)) / ball_df['D'].values**2
                    theo_ratio = ( np.pi + (disk_df['D'].values * 2)) / ball_df['D'].values**2
                    print(f"Theoretical ratio for Line: {theo_ratio}")
                    print(f"Disk diameters: {disk_df['D'].values}, Ball diameters: {ball_df['D'].values}")
                    label = r"$\Delta_{Line}/\Delta_{ball}$"
                else:
                    raise ValueError(f"Unknown reference shape: {reference_shape}")
                theo_handle, = plt.plot(
                    common_diam, theo_ratio, '--', color='gray',
                    label=label
                )
            if reference == "surface":
                if reference_shape == "Line":
                    theo_ratio = (6 * np.sqrt(np.pi) * (disk_df['surface'] / 2 - (2/3) * np.pi)) / (disk_df['surface'] ** 1.5)
                    label = r"$V_{line}/V_{ball}$"
                    print(theo_ratio)

                else:
                    raise ValueError(f"Unknown reference shape: {reference_shape}")
                theo_handle, = plt.plot(
                    common_diam, theo_ratio, '--', color='gray',
                    label=label
                )

    handles = mu_handles + ([theo_handle] if theo_handle is not None else [])
    labels = [h.get_label() for h in handles]

    plt.axhline(1.0, color='gray', linestyle=':', linewidth=1)

    reference_symbol = r"$\Delta$" if reference == "surface" else r"$D_\text{Line}$"

    if reference == "surface":
        plt.xlabel(rf"Surface {reference_symbol}")
    if reference == "DiskDiameter":
        plt.xlabel(rf"Line diameter - {reference_symbol}")
    if reference_shape == "Disk":
        ylabel = r"$t_{detect}^{Ball}(X^\mu)/t_{detect}^{Disk}(X^\mu)$"
    elif reference_shape == "Line":
        ylabel = r"$t_{detect}^{Ball}(X^\mu)/t_{detect}^{Line}(X^\mu)$"
    else:
        raise ValueError(f"Unknown reference shape: {reference_shape}")
    plt.ylabel(ylabel)
    plt.ylim(y_min, y_max)
    plt.grid(True)
    plt.legend(handles, labels, ncol=2)

    plt.tight_layout()

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, bbox_inches="tight")
    
    plt.show()
    plt.close()



In [ ]:
# file_paths = glob.glob('./results/fixed_volume/*')
# df_list = [pd.read_csv(file) for file in file_paths if os.path.isfile(file) ]
# df = pd.concat(df_list, ignore_index=True)
df = pd.read_csv('./results/fixed_volume/review/detection_time_fixed_volume.csv')
print(f"Surface values: {df['D'].unique()}")


plot_detection_time_ratio(df,n_walkers=1,n_targets=1,fixed_target_dist=0,probability=1.0, reference_shape = 'Line',mu_list=[1.0,1.2,1.4,1.6,1.8,2.0,2.2,2.4,2.6,2.8,3.0], output_path='./plots/pdf_figures/detection_time_ratio_ball_line.pdf')
plot_detection_time_ratio(df,n_walkers=1,n_targets=1,fixed_target_dist=0,probability=1.0, reference_shape = 'Line',mu_list=[1.0,1.2,1.4,1.6,1.8,2.0,2.2,2.4,2.6,2.8,3.0], output_path='./plots/pdf_figures/detection_time_ratio_ball_line_errorbands.pdf', error_bands=True)


In [ ]:
# NB: the per-thread shards concatenate to exactly the aggregate CSV, so globbing
# the whole directory double-counts every sample and understates the SEM by sqrt(2).
df = pd.read_csv('./results/detection_time_fixed_surface/review/detection_time_fixed_surface.csv')
print(f"Surface values: {df['D'].unique()}")


plot_detection_time_ratio(df,n_walkers=1,n_targets=1,fixed_target_dist=0,probability=1.0, reference_shape = 'Line',mu_list=[1.0,1.2,1.4,1.6,1.8,2.0,2.2,2.4,2.6,2.8,3.0], reference='surface', output_path='./plots/pdf_figures/detection_time_ratio_ball_line_surface.pdf')
plot_detection_time_ratio(df,n_walkers=1,n_targets=1,fixed_target_dist=0,probability=1.0, reference_shape = 'Line',mu_list=[1.0,1.2,1.4,1.6,1.8,2.0,2.2,2.4,2.6,2.8,3.0], reference='surface', output_path='./plots/pdf_figures/detection_time_ratio_ball_line_surface_errorbands.pdf', error_bands=True)


# Figure 4 (c + d)

In [ ]:
def plot_detection_time_vs_delta_by_mu_facets(
    df,
    output_dir=None,
    filename="detection_time_vs_delta_by_surface_grid.png",
    mu_filter=[2.0, 2.5, 3.0],
    surface_filter=[50, 100, 200, 300, 400, 500],  # now includes 200
    error_bands=False
):
    """
    Creates a 2x2 grid of small multiples: each subplot corresponds to a surface D.
    Each subplot shows detection_time vs delta for different μ.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Must contain 'delta', 'detection_time', 'mu', 'D'
    output_dir : str
        Directory where the plot will be saved
    filename : str
        Output filename
    mu_filter : list of floats
        μ values to include
    surface_filter : list of floats or ints
        D values to include
    """
    required_cols = {"delta", "detection_time", "mu", "D"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Filter dataframe
    df = df[df['mu'].isin(mu_filter)]
    if surface_filter is not None:
        df = df[df['D'].isin(surface_filter)]
    if df.empty:
        print("No data to plot after filtering!")
        return

    # Average detection_time per (mu, delta, D)
    grouped = (
        df.groupby(["mu", "delta", "D"])["detection_time"]
        .agg(detection_time='mean', detection_time_std='std', detection_time_count='count')
        .reset_index()
        .sort_values("delta")
    )

    surfaces = sorted(grouped['D'].unique())
    n_surfaces = len(surfaces)
    # Determine grid layout: 2x2
    n_rows = 2
    n_cols = 2

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4*n_cols, 3*n_rows),
        sharey=True
    )

    # Flatten axes array for easy iteration
    axes = axes.flatten()
    print(axes)
    # Colors for different μ
    #cmap = plt.cm.viridis
    #mu_vals = sorted(grouped['mu'].unique())
    #colors = {mu: cmap(i / (len(mu_vals)-1)) for i, mu in enumerate(mu_vals)}

    # With this:
    mu_vals = sorted(grouped['mu'].unique())
    palette = ["tab:blue", "tab:orange", "tab:green"]
    colors = {mu: palette[i] for i, mu in enumerate(mu_vals)}

    for ax, D_val in zip(axes, surfaces):
        sub_D = grouped[grouped['D'] == D_val]
        for mu_val, sub_mu in sub_D.groupby("mu"):
            ax.plot(
                sub_mu["delta"],
                sub_mu["detection_time"],
                marker="o",
                color=colors[mu_val],
                label=rf"$\mu={mu_val}$"
            )
            if error_bands:
                sem = sub_mu["detection_time_std"] / np.sqrt(sub_mu["detection_time_count"])
                ax.fill_between(
                    sub_mu["delta"],
                    sub_mu["detection_time"] - sem,
                    sub_mu["detection_time"] + sem,
                    color=colors[mu_val], alpha=0.2, linewidth=0
                )
        ax.set_title(f"$\\Delta={D_val}$")
        ax.set_xlabel(r"$\delta$")
        ax.grid(True)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))

        # set max y limit
        #ax.set_ylim(0, grouped['detection_time'].max()*1.3)

    # Hide any unused subplots (if surface_filter < 4)
    for ax in axes[n_surfaces:]:
        ax.axis('off')

    axes[0].set_ylabel(r"$t^{x^\mu}_{\mathrm{detect}}(S)$")
    axes[2].set_ylabel(r"$t^{x^\mu}_{\mathrm{detect}}(S)$")

    # Add a single legend outside the plots
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=len(mu_vals), bbox_to_anchor=(0.5, 1.05))

    plt.tight_layout(rect=[0, 0, 1, 0.95])  # leave space for legend

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, filename)
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved plot to {output_path}")
    
    plt.show()
    plt.close()
    

In [ ]:
#file_paths = glob.glob('./results/detection_time_small_delta/review/detection_time_small_delta_large_mu.csv*')
#df_list = [pd.read_csv(file) for file in file_paths if os.path.isfile(file) ]
#df = pd.concat(df_list, ignore_index=True)
df = pd.read_csv('./results/detection_time_small_delta/review/detection_time_small_delta_large_mu.csv')   
plot_detection_time_vs_delta_by_mu_facets(df, filename="detection_time_vs_delta_by_mu.pdf", mu_filter = [2.0, 2.5, 3.0], surface_filter=[200, 300, 400, 500], output_dir='./plots/pdf_figures/')
plot_detection_time_vs_delta_by_mu_facets(df, filename="detection_time_vs_delta_by_mu_errorbands.pdf", mu_filter = [2.0, 2.5, 3.0], surface_filter=[200, 300, 400, 500], output_dir='./plots/pdf_figures/', error_bands=True)

In [ ]:

df = pd.read_csv('./results/detection_time_small_delta/review/detection_time_small_delta_small_mu.csv')   


plot_detection_time_vs_delta_by_mu_facets(df, filename="detection_time_vs_delta_by_mu_small.pdf", mu_filter = [1.0, 1.5, 2.0], surface_filter=[200, 300, 400, 500], output_dir='./plots/pdf_figures/')
plot_detection_time_vs_delta_by_mu_facets(df, filename="detection_time_vs_delta_by_mu_small_errorbands.pdf", mu_filter = [1.0, 1.5, 2.0], surface_filter=[200, 300, 400, 500], output_dir='./plots/pdf_figures/', error_bands=True)

# Other experiments

In [ ]:
def plot_detection_vs_mu_for_all_surface(df, fixed_n,
                                   fixed_n_volume=None,
                                   fixed_n_targets=None,
                                   fixed_probability=None,
                                   target_shapes=None,
                                   save_path=None,
                                   parameters = None,
                                   error_bands=False):
    # Base filtering
    base_filter = (df['n_walkers'] == fixed_n)
    if fixed_n_volume is not None:
        base_filter &= (df['n_volume'] == fixed_n_volume)
    if fixed_n_targets is not None:
        base_filter &= (df['n_targets'] == fixed_n_targets)
    if fixed_probability is not None:
        base_filter &= (df['probability'] == fixed_probability)  # Considera solo D >= 2  
    if parameters is not None:
        base_filter &= df['surface'].isin(parameters)
    else:
        base_filter &= (df['surface'] > 100)
    subset = df[base_filter].copy()

    # Filter shapes if provided
    if target_shapes is not None:
        subset = subset[subset['TargetShape'].isin(target_shapes)]

    # Grouping
    group_cols = ['mu', 'surface', 'TargetShape', 'n_walkers']
    if fixed_n_volume is not None:
        group_cols.append('n_volume')
    if fixed_n_targets is not None:
        group_cols.append('n_targets')
    if fixed_probability is not None:
        group_cols.append('probability')

    stats = subset.groupby(group_cols)['detection_time'].agg(['mean', 'std', 'count']).reset_index()
    stats = stats.rename(columns={'mean': 'detection_time_mean', 'std': 'detection_time_std', 'count': 'detection_time_count'})

    plt.figure(figsize=(8, 6))    

    # Define line styles for shapes (max 2 shapes)
    line_styles = ['-', '--']

    # Get shape list (for legend later)
    unique_shapes = sorted(stats['TargetShape'].unique())
    shape_style_map = {shape: line_styles[i % len(line_styles)] for i, shape in enumerate(unique_shapes)}

    # Create color palette for each diameter
    unique_surface = sorted(stats['surface'].unique())
    colors = plt.cm.tab10.colors  # Enough distinct colors

    # Plotting
    for color_idx, surface_val in enumerate(unique_surface):
        dia_data = stats[stats['surface'] == surface_val]
        for shape in unique_shapes:
            shape_data = dia_data[dia_data['TargetShape'] == shape].sort_values('mu')
            if shape_data.empty:
                continue
            plt.plot(shape_data['mu'],
                     shape_data['detection_time_mean'],
                     color=colors[color_idx % len(colors)],
                     linestyle=shape_style_map[shape],
                     marker='o')
            if error_bands:
                sem = shape_data['detection_time_std'] / np.sqrt(shape_data['detection_time_count'])
                plt.fill_between(shape_data['mu'],
                                  shape_data['detection_time_mean'] - sem,
                                  shape_data['detection_time_mean'] + sem,
                                  color=colors[color_idx % len(colors)], alpha=0.2, linewidth=0)
            

    # Legend for diameters (colors)
    # Legend for diameters (colors)
    surface_handles = [
        plt.Line2D([0], [0], color=colors[i % len(colors)], lw=2)
        for i in range(len(unique_surface))
    ]
    surface_labels = [f"{d}" for d in unique_surface]

    # Legend for shapes (line styles)
    if len(unique_shapes) > 1:
        shape_handles = [
            plt.Line2D([0], [0], color='black', linestyle=shape_style_map[s], lw=2)
            for s in unique_shapes
        ]
        def get_title(shape):
            if shape == "Ball":
                return "Ball"
            elif shape == "Ball_boundary":
                return "Boundary"
            else:
                return shape
        shape_labels = [get_title(s) for s in unique_shapes]
    else:
        shape_handles = []
        shape_labels = []

    # Merge legends
    all_handles = surface_handles + shape_handles
    all_labels = surface_labels + shape_labels

    legend = plt.legend(
        all_handles,
        all_labels,
        framealpha=0.5,
        loc="upper right",
        ncol=4,
        columnspacing=0.8,
    )


    plt.gca().add_artist(legend)  # Keep both legends

    # Axis and title
    plt.xticks(sorted(stats['mu'].unique()))
    plt.xlabel(r'$\mu$')
    plt.yscale('log')
    plt.ylabel(r'$t_{detect}(X^{\mu})$')

    titolo = f'Detection Time vs μ'
    if fixed_n_volume is not None:
        side = round(fixed_n_volume ** (1/3))
        titolo += f', side={side}'
    if fixed_n_targets is not None:
        titolo += f', n_targets={fixed_n_targets}'
    if fixed_probability is not None:
        titolo += f', p={fixed_probability}'
    titolo += ')'
    #plt.title(titolo)

    plt.grid(True)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
    
    plt.show()
    plt.close()


In [ ]:
df = pd.read_csv('./results/detection_time_mu/review/detection_time_mu_ball_line.csv')

plot_detection_vs_mu_for_all_surface(df, fixed_n=1, fixed_n_volume=134217728, fixed_probability=1, fixed_n_targets=1, target_shapes=['Ball'], parameters = [8,16,32,64,128, 256, 512, 1024, 2048, 4096, 8000], save_path='./plots/pdf_figures/detection_time_vs_mu_ball.pdf')
plot_detection_vs_mu_for_all_surface(df, fixed_n=1, fixed_n_volume=134217728, fixed_probability=1, fixed_n_targets=1, target_shapes=['Ball'], parameters = [8,16,32,64,128, 256, 512, 1024, 2048, 4096, 8000], save_path='./plots/pdf_figures/detection_time_vs_mu_ball_errorbands.pdf', error_bands=True)

In [ ]:
def plot_ratio_detection_time_vs_xratio_surface_grid(
    df,
    mu_list,
    surface_list,
    baseline_mu=2.0,
    group_cols=None,
    agg="mean",
    xscale="linear",
    yscale="linear",
    ncols=3,
    error_bands=False,
):
    if group_cols is None:
        group_cols = [
            "n_walkers", "n_volume", "lmax", "surface", "TargetShape",
            "n_targets", "fixed_target_dist", "probability",
            "surface_selector", "first_touch_steps", "second_touch_steps",
            "delta_selector", "delta",
        ]

    required = set(group_cols) | {"mu", "surface", "delta", "detection_time"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    if baseline_mu not in mu_list:
        raise ValueError(f"mu_list must include baseline_mu={baseline_mu}.")

    dff = df[df["mu"].isin(mu_list)].copy()
    dff = dff[dff["surface"].isin(surface_list)].copy()  # filter by list via isin [web:63]
    if dff.empty:
        raise ValueError("No data after filtering.")

    dff["surface"] = dff["surface"].astype(float)
    dff["delta"] = dff["delta"].astype(float)
    dff["x2"] = np.sqrt(dff["surface"])
    dff["x1"] = dff["x2"] ** dff["delta"]
    dff["x_ratio"] = dff["x1"] / dff["x2"]

    keys = group_cols + ["mu"]
    if agg == "mean":
        g = dff.groupby(keys, as_index=False)["detection_time"].agg(
            detection_time="mean", detection_time_std="std", detection_time_count="count"
        )
    elif agg == "median":
        g = dff.groupby(keys, as_index=False)["detection_time"].agg(
            detection_time="median", detection_time_std="std", detection_time_count="count"
        )
    else:
        raise ValueError("agg must be 'mean' or 'median'.")

    g["surface"] = g["surface"].astype(float)
    g["delta"] = g["delta"].astype(float)
    g["x2"] = np.sqrt(g["surface"])
    g["x1"] = g["x2"] ** g["delta"]
    g["x_ratio"] = g["x1"] #g["x1"] / g["x2"]

    surfaces = [float(s) for s in surface_list]
    n = len(surfaces)
    ncols = max(1, int(ncols))
    nrows = int(math.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, sharex=True, sharey=True)  # sharex/sharey supported [web:54]
    axes = np.array(axes).reshape(-1)

    for i, srf in enumerate(surfaces):
        ax = axes[i]
        subS = g[g["surface"] == srf].copy()

        base = subS[subS["mu"] == baseline_mu].copy()
        oth = subS[(subS["mu"] != baseline_mu) & (subS["mu"] < baseline_mu)].copy()

        join_cols = group_cols + ["x_ratio"]
        base = base[join_cols + ["detection_time", "detection_time_std", "detection_time_count"]].rename(
            columns={"detection_time": "t_base", "detection_time_std": "t_base_std", "detection_time_count": "t_base_count"}
        )
        oth = oth[join_cols + ["mu", "detection_time", "detection_time_std", "detection_time_count"]].rename(
            columns={"detection_time": "t_mu", "detection_time_std": "t_mu_std", "detection_time_count": "t_mu_count"}
        )

        merged = pd.merge(oth, base, on=join_cols, how="inner")
        if merged.empty:
            ax.set_title(f"surface={srf:g} (no matches)")
            ax.grid(True, which="both", alpha=0.2)
            continue

        merged["ratio"] =  merged["t_mu"] / merged["t_base"]
        if error_bands:
            sem_mu = merged["t_mu_std"] / np.sqrt(merged["t_mu_count"])
            sem_base = merged["t_base_std"] / np.sqrt(merged["t_base_count"])
            rel_err = np.sqrt((sem_mu / merged["t_mu"])**2 + (sem_base / merged["t_base"])**2)
            merged["ratio_err"] = merged["ratio"] * rel_err

        for mu_val, smu in merged.groupby("mu", sort=True):
            if error_bands:
                s = smu.groupby("x_ratio", as_index=False).agg(
                    ratio=("ratio", "median"), ratio_err=("ratio_err", "median")
                ).sort_values("x_ratio")
            else:
                s = smu.groupby("x_ratio", as_index=False)["ratio"].median().sort_values("x_ratio")
            line, = ax.plot(s["x_ratio"], s["ratio"], marker="o", linewidth=1.4, label=f"$\\mu={mu_val:g}$")
            if error_bands:
                ax.fill_between(s["x_ratio"], s["ratio"] - s["ratio_err"], s["ratio"] + s["ratio_err"],
                                 color=line.get_color(), alpha=0.2, linewidth=0)

        value = int(math.log(srf, 2))
        ax.set_title(rf"$\Delta_B=2^{{{value}}}$")
        ax.set_xscale(xscale)
        ax.set_yscale(yscale)
        ax.grid(True, which="both", alpha=0.25)
        if i == 0:
            ax.set_xlim(0, 35)


    for j in range(n, len(axes)):
        axes[j].axis("off")

    for ax in axes[:n]:
        ax.label_outer()

    axes[0].set_ylabel(r"$\frac{t_{\mathrm{detect}}(X^\mu)}{t_{\mathrm{detect}}(X^2)}$")
    axes[2].set_ylabel(r"$\frac{t_{\mathrm{detect}}(X^\mu)}{t_{\mathrm{detect}}(X^2)}$")
    for ax in axes[max(0, n - ncols):n]:
        ax.set_xlabel(r"$x_1$")
        xmin, xmax = ax.get_xlim()
        #ax.set_xticks([xmin, xmax], labels=["1", r"$\sqrt{\Delta_B}/2$"])
        ax.tick_params(axis="x", which="major", labelsize=10)

    handles, labels = None, None
    for ax in axes[:n]:
        h, l = ax.get_legend_handles_labels()
        if h:
            handles, labels = h, l
            break
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(len(handles), 6))

    fig.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()  # show the grid once [web:75]
    return fig, axes[:n]



In [ ]:
# Read only the merged aggregate; including per-thread shards would double-count samples.
df = pd.read_csv("./results/detection_time_ratio_fixed_surface/detection_time_ratio_fixed_surface.csv")

plot_ratio_detection_time_vs_xratio_surface_grid(
    df,
    [1.0,1.2,1.4,1.6,1.8,2.0],
    [1024,4096,16384,65536],
    baseline_mu=2.0,
    group_cols=None,
    agg="mean",          # "mean" or "median"
    xscale="linear",
    yscale="linear",
    ncols=2,
)
plot_ratio_detection_time_vs_xratio_surface_grid(
    df,
    [1.0,1.2,1.4,1.6,1.8,2.0],
    [1024,4096,16384,65536],
    baseline_mu=2.0,
    group_cols=None,
    agg="mean",          # "mean" or "median"
    xscale="linear",
    yscale="linear",
    ncols=2,
    error_bands=True,
)